# Exploration de la méthode de calcul de points par conv2D

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import torch as t
from pytorch_lightning.trainer import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar
from pytorch_lightning.loggers import TensorBoardLogger
from memory_profiler import profile

from data_handling.database_service import DatabaseService
from data_handling.conv_method_data_prep import ConvMethodDataPreparator
from points_methods.conv2D_model.net import CompetitionAutoencoderNet
from points_methods.conv2D_model.data_module import CompetitionDataModule
from points_methods.conv2D_model.batch_handling import random_crop_batch_to_smallest

## Préparation des données

Sauvegarde des données préparées sous forme de fichiers HDF5 pour ne pas surcharger la RAM lors de l'entraînement

In [3]:
files_saving_path = Path.cwd() / "hdf5_data" / "1y_tables"
db_service = DatabaseService()
data_preparator = ConvMethodDataPreparator(
    db_service, competition_table_period=timedelta(days=365)
)
# ONLY ONCE
# data_preparator.save_competition_data_over_period(
#     starting_date=datetime(2001, 1, 1),
#     ending_date=datetime(2025, 3, 1),
#     saving_path=files_saving_path,
#     seed=0,
# )

## Entraînement du modèle

In [ ]:
saving_path = Path.cwd() / "checkpoints" / "local" / "test_V2enc&dec"
db_service = DatabaseService()
data_preparator = ConvMethodDataPreparator(db_service)

max_epochs = 20
batch_size = 4
competition_table_period = timedelta(days=365)
train_test_lim = datetime(2023, 1, 1)
hidden_part = 0.15
num_workers = 4

net_hyperparams = dict(
    in_channels_enc=6,
    hidden_channels_enc=16,
    nb_hidden_layers_enc=3
    zdim_line=4,
    zdim_col=4,
    #nb_channel_transformations_enc=3,
    #nb_hidden_layers_per_channel_transform_enc=3,
    hidden_channels_dec=16,
    nb_hidden_layers_dec=3,
    huber_loss_delta=5 / 90,  # loss linéaire à partir de 10 secondes d'erreur
)

checkpoint_callback = ModelCheckpoint(
    monitor="Median prediction error (s)",
    mode="min",
    save_top_k=1,
    dirpath=saving_path.parent,
    filename=saving_path.name,
)
tqdm_bar = TQDMProgressBar(refresh_rate=1)
logger = TensorBoardLogger(
    save_dir=Path.cwd() / "lightning_logs", name=saving_path.name
)
trainer = Trainer(
    callbacks=[checkpoint_callback, tqdm_bar],
    max_epochs=max_epochs,
    log_every_n_steps=1,
    logger=logger,
)

data_module = CompetitionDataModule(
    files_saving_path,
    batch_size=batch_size,
    competition_table_period=competition_table_period,
    train_test_lim=train_test_lim,
    hidden_part=hidden_part,
    num_workers=num_workers,
)

model = CompetitionAutoencoderNet(**net_hyperparams)

trainer.fit(model, data_module)


# model.load_data(
#     starting_date_train=starting_date_train,
#     ending_date_train=ending_date_train,
#     starting_date_val=starting_date_val,
#     ending_date_val=ending_date_val,
# )

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


TypeError: CompetitionAutoencoderNet.__init__() got an unexpected keyword argument 'nb_channel_transformations_enc'

In [ ]:
%debug

In [ ]:
raw_batch = []
for i in range(16):
    raw_batch.append(
        (
            t.rand(121, 3108),
            t.rand(3108, 5) > 0.5,
            t.rand(121, 3108) > 0.5,
            t.rand(121, 3108) > 0.5,
        )
    )
random_crop_batch_to_smallest(raw_batch)

In [ ]:
from torch.utils.data import DataLoader

from points_methods.conv2D_model.batch_handling import random_crop_batch_to_smallest

train_dataloader = DataLoader(
    model.train_dataset,
    batch_size=4,
    collate_fn=random_crop_batch_to_smallest,
    num_workers=4,
)
val_dataloader = DataLoader(
    model.val_dataset,
    batch_size=4,
    collate_fn=random_crop_batch_to_smallest,
    num_workers=4,
)

In [ ]:
import torch

x, m = next(iter(train_dataloader))
res = model.net(x, m)
res.shape

In [ ]:
(m.sum(axis=1) == 1).sum()

In [ ]:
res[:, :, :, :]

In [ ]:
%debug

# Brouillon

In [ ]:
db_service = DatabaseService()
preparator = ConvMethodDataPreparator(db_service)
res = preparator.get_comp_dataframes_at_specific_date_phase(
    table_end_date=datetime(2015, 2, 1), phase="finale"
)

In [ ]:
res[0].columns

In [ ]:
db_service = DatabaseService()
preparator = ConvMethodDataPreparator(db_service)
res = preparator.get_competition_tensors_over_period(
    starting_date=datetime(2014, 1, 1), ending_date=datetime(2015, 2, 1)
)

In [ ]:
data, mask = res[0]
data_t = t.from_numpy(data.copy()).unsqueeze(0)
mask_t = t.from_numpy(mask.copy()).unsqueeze(0)
net = CompetitionAutoencoderNet(
    in_channels_enc=6,
    hidden_channels_enc=16,
    zdim_line=4,
    zdim_col=4,
    nb_channel_transformations_enc=3,
    nb_hidden_layers_per_channel_transform_enc=3,
    hidden_channels_dec=16,
    nb_hidden_layers_dec=3,
    huber_loss_delta=10 / 90,  # loss linéaire à partir de 10 secondes d'erreur
)
net(data_t, mask_t)

In [ ]:
raw = [(1, 2) for _ in range(10)]
ones, twos = zip(*raw)
print(ones)
print(twos)